In [1]:
import torch

## 1 ReLU

In [5]:
# ✏️ YOUR IMPLEMENTATION HERE

def relu(x: torch.Tensor) -> torch.Tensor:
    return x*(x>0) # Булев тензор формы x [1,2,3,-1] -> [1,1,1,0]*x  = [1,2,3,0]

def backward(x: torch.Tensor):
    return (x>0)

In [ ]:
# 🧪 Test your implementation (feel free to add more debug prints)
x = torch.tensor([-2., -1., 0., 1., 2.])
print("Input: ", x)
print("Output:", relu(x))
print("Shape: ", relu(x).shape)

## 2 Softmax

In [7]:
# ✏️ YOUR IMPLEMENTATION HERE

def my_softmax(x: torch.Tensor, dim: int = -1) -> torch.Tensor:
    x= x - x.max(dim=dim,keepdim=True).values
    e = x.exp()

    return e/e.sum(dim=dim,keepdim=True)

In [8]:
x = torch.tensor([1.0, 2.0, 3.0])
x - x.max(dim=-1,keepdim=True).values
e = x.exp()
e/e.sum(dim=-1,keepdim=True)

tensor([0.0900, 0.2447, 0.6652])

## 16 Cross-Entropy Loss

In [16]:
# logits входные даныне и таргеты подсчитанные
# targets - true таргеты
def cross_entropy_loss(logits, targets):
    log_probs = logits - torch.logsumexp(logits,dim=-1,keepdim=True)
    n = logits.shape[0]
    print(log_probs)
    print(log_probs[torch.arange(n),targets],targets)
    return -log_probs[torch.arange(n),targets].mean() # продвинутая индексация
    
# logsumexp = log(sum(exp())) - внутри себя вычитает максимум

In [17]:
# 🧪 Debug
logits = torch.randn(4, 10)
targets = torch.randint(0, 10, (4,))
print('Loss:', cross_entropy_loss(logits, targets))
print('Ref: ', torch.nn.functional.cross_entropy(logits, targets))

tensor([[-3.3594, -3.3950, -3.8165, -1.3267, -2.5958, -2.7821, -1.7747, -2.3348,
         -2.2143, -2.0229],
        [-4.4624, -2.7354, -1.4034, -2.4898, -1.4009, -1.8208, -2.4950, -2.7657,
         -3.4057, -4.8226],
        [-1.9623, -2.1178, -1.9926, -2.0765, -2.4584, -2.3440, -2.3583, -2.5816,
         -2.8265, -2.7107],
        [-1.6967, -2.3694, -1.8685, -2.8422, -1.9689, -2.2651, -2.5222, -3.1053,
         -2.8826, -2.4532]])
tensor([-1.7747, -2.7354, -2.7107, -2.2651]) tensor([6, 1, 9, 5])
Loss: tensor(2.3715)
Ref:  tensor(2.3715)


In [9]:
torch.arange(10)

tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])

## 17 Dropout

In [18]:
import torch
import torch.nn as nn

In [19]:
# ✏️ YOUR IMPLEMENTATION HERE

class MyDropout(nn.Module):
    def __init__(self, p=0.5):
        super().__init__() # вызов конструктора родителя
        self.p = p

    def forward(self, x):
        if not self.training or self.p ==0.0:
            return x
        mask = (torch.rand_like(x) >= self.p).to(x.dtype) # нормальное распрелеление 

        return x * mask / (1.0 - self.p)

In [20]:
# 🧪 Debug
d = MyDropout(p=0.5)
d.train()
x = torch.ones(10)
print('Train:', d(x))
d.eval()
print('Eval: ', d(x))

Train: tensor([2., 0., 2., 2., 2., 2., 0., 0., 2., 2.])
Eval:  tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])


## 3 Linear Regression

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

class LinearRegression:
    def closed_form(self, X: torch.Tensor, y: torch.Tensor):
        """Normal equation: w = (X^T X)^{-1} X^T y"""
        N,D = X.shape
        Xb = torch.cat([X,torch.ones(N,1)],dim=1)

        #torch.linald.lstsq: min||Xb * theta - y||  -> QR/SVD

        theta = torch.linalg.lstsq(Xb,y.unsqueeze(1)).solution.squeeze(1)
        print(theta)
        return theta[:D],theta[D]

    def gradient_descent(self, X: torch.Tensor, y: torch.Tensor,
                         lr: float = 0.01, steps: int = 1000):
        """Manual gradient descent loop"""
        N,D = X.shape
        w = torch.zeros(D)
        b = torch.zeros(())

        for _ in range(steps):
            err = (X @ w+b) - y

            grad_W = (2.0 / N) * (X.T @ err) # d/dw( 1/N * sum((X @ w+b) - y)^2 ) - mse 
            grad_b = (2.0\N) * (err.sum())

            w-= lr*grad_W
            b-= lr*grad_b

        return w,b

    def nn_linear(self, X: torch.Tensor, y: torch.Tensor,
                  lr: float = 0.01, steps: int = 1000):
        """Train nn.Linear with autograd"""
        N,D = X.shape

        model = nn.Linear(D,1)
        opt = torch.optim.SGD(model.parameters(),lr=lr)

        loss_fn = nn.MSELoss()
        y_proc = y.view(-1,1) # из [100] в [100,1]
        for _ in range(steps):
            opt.zero_grad()
            loss_fn(model(X),y_proc).backward()
            opt.step()

        print(model.weight)
        return model.weight.detach().reshape(-1),model.bias.detach().reshape(())


In [44]:
# 🧪 Debug
torch.manual_seed(42)
X = torch.randn(100, 3)
true_w = torch.tensor([2.0, -1.0, 0.5])
y = X @ true_w + 3.0

model = LinearRegression()

w_cf, b_cf = model.closed_form(X, y)
print(f"Closed-form:  w={w_cf}, b={b_cf.item():.4f}")

w_gd, b_gd = model.gradient_descent(X, y, lr=0.05, steps=2000)
print(f"Grad descent: w={w_gd}, b={b_gd.item():.4f}")

w_nn, b_nn = model.nn_linear(X, y, lr=0.05, steps=2000)
print(f"nn.Linear:    w={w_nn}, b={b_nn.item():.4f}")

print(f"\nTrue:         w={true_w}, b=3.0")

tensor([ 2.0000, -1.0000,  0.5000,  3.0000])
Closed-form:  w=tensor([ 2.0000, -1.0000,  0.5000]), b=3.0000
Grad descent: w=tensor([ 2.0000, -1.0000,  0.5000]), b=3.0000
Parameter containing:
tensor([[ 2.0000, -1.0000,  0.5000]], requires_grad=True)
nn.Linear:    w=tensor([ 2.0000, -1.0000,  0.5000]), b=3.0000

True:         w=tensor([ 2.0000, -1.0000,  0.5000]), b=3.0


In [ ]:
class MLPClassifier(nn.Module):
    def __init__(self,input_size,hiden_size = [512,256,128],dropout_rate = 0.2): # input_size = 784
        super(MLPClassifier,self).__init__()
        layers = []
        prev_size = input_size
        for size in hiden_size:
            layers.append(nn.Linear(prev_size,size))
            layers.append(nn.BatchNorm1d(size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            prev_size = size

        layers.append(nn.Linear(prev_size,10))
        self.network = nn.Sequential(*layers)

    def forward(self,x):
        return self.network(x)

## 22 Conv2D

In [4]:
import torch
import torch.nn.functional as F

In [3]:
# ✏️ YOUR IMPLEMENTATION HERE

def my_conv2d(x, weight, bias=None, stride=1, padding=0):
    # NCHW 
    N,C,H,W = x.shape
    O, C_weight , kh,kw = weight.shape
    
    assert C == C_weight

    if padding > 0: # Проверяем, нужно ли добавлять padding
        x = F.pad(x, (padding,padding,padding,padding)) # padding с каждой стороны

    Ho = (x.shape[2] - kh) // stride + 1 # выходные размеры conv2d
    Wo = (x.shape[3] - kw) // stride + 1

    cols = F.unfold(x,(kh,kw),stride=stride) # -> (N, C*kh*kw,L)

    out = torch.einsum('oc,ncl->nol',weight.reshape(O,-1),cols)
    out = out.reshape(N,O,Ho,Wo)

    if bias is not None:
        out = out + bias.view(1,-1,1,1)
    return out 



In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def my_conv2d(x, weight, bias=None, stride=1, padding=0):

    #NCHW - 
    N, C, H, W = x.shape
    O, _, kh, kw = weight.shape

    if padding > 0:
        x = F.pad(x, (padding, padding, padding, padding))
    
    Ho = (x.shape[2] - kh) // stride + 1
    Wo = (x.shape[3] - kw) // stride + 1

    #im2col: (c, m, n), c in C, m in kh, n in kw -> C*kh*kw

    #cols = F.unfold(x, (kh, kw), stride=stride) # -> (N, C*kh*kw, L)
   
    #out = torch.einsum("oc,ncl->nol", weight.reshape(O, -1), cols)

    out = torch.zeros(N, O, Ho, Wo, dtype=x.dtype)
    for n in range(N):
        for o in range(O):
            for i in range(Ho):
                for j in range(Wo):
                    s = 0.0
                    for c in range(C):
                        for m in range(kh):
                            for u in range(kw):
                                s += weight[o, c, m, u] * x[n, c, i*stride + m, stride*j + u]
                    out[n, o, i, j] = s
    
            if bias is not None:
                out[n, o] += bias[o]
    return out


In [ ]:
results = model.train(
    data =  'C:/Users/mikhail.zanochuev/Desktop/Repository/Data-Science-Course/Поступашки\Лекция 2\output_photos_rus_signes\data.yaml'
    ,epochs = 30
    ,imagesz = 640
    ,batch = 32
    ,device = 0
    ,project = 'C:\Users\mikhail.zanochuev\Desktop\Repository/Data-Science-Course\Поступашки\Лекция 2\runs'
    , name = 'rus_signs'

)   

## 26 LoRA

In [13]:
import torch
import torch.nn as nn
# ✏️ YOUR IMPLEMENTATION HERE

class LoRALinear(nn.Module):
    def __init__(self, in_features, out_features, rank, alpha=1.0):
        super().__init__()
        self.linear = nn.Linear(in_features,out_features)
        self.linear.weight.requires_grad = False

        if self.linear.bias is not None:
            self.linear.bias.requires_grad = False

        # Инициализация B всегда 0, а A инициализируем kaiming, B@A = 0
        self.lora_A = nn.Parameter(torch.empty(rank, in_features))
        self.lora_B = nn.Parameter(torch.zeros(out_features,rank))

        nn.init.kaiming_uniform_(self.lora_A,a = 5 ** 0.5)
        self.scaling = alpha/rank
    def forward(self, x):
        return self.linear(x) + (x@self.lora_A.T@self.lora_B.T) * self.scaling

In [14]:
# 🧪 Debug
layer = LoRALinear(16, 8, rank=4)
x = torch.randn(2, 16)
print('Output:', layer(x).shape)
print('Trainable:', sum(p.numel() for p in layer.parameters() if p.requires_grad))
print('Total:    ', sum(p.numel() for p in layer.parameters()))

Output: torch.Size([2, 8])
Trainable: 96
Total:     232


In [15]:
layer = LoRALinear(10, 5, rank=2)

for name, param in layer.named_parameters():
    print(name, param.requires_grad)


lora_A True
lora_B True
linear.weight False
linear.bias False


In [16]:
# ✅ SUBMIT
from torch_judge import check
check('lora')


🧪 Testing: LoRA (Low-Rank Adaptation) (Medium)
──────────────────────────────────────────────────
  ✅ [1/5] Base weights frozen (0.9ms)
  ✅ [2/5] LoRA parameter shapes (0.5ms)
  ✅ [3/5] B=0 means output equals base (8.6ms)
  ✅ [4/5] Only LoRA params get gradients (1.3ms)
  ✅ [5/5] Forward computation (6.2ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (17.7ms total)
  Progress saved. Run status() to see your dashboard.



## 5 Softmax Attention

Причина: при больших d_k скалярные произведения становятся большими, softmax попадает в зону насыщения (градиенты близки к 0). Масштабирование возвращает значения в диапазон с нормальной дисперсией.

In [ ]:
import torch
import math 
# ✏️ YOUR IMPLEMENTATION HERE

def scaled_dot_product_attention(Q, K, V):
    # Q (Query)    = "Что я ищу?"
    # K (Key)      = "Что у меня есть?" (метки)
    # V (Value)    = "Какая информация?" (содержимое)
    d_k = Q.shape[-1]                              # размерность ключей
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)  # сходство Q и K
    weights = torch.softmax(scores, dim=-1)        # нормализация → вероятности
    return weights @ V                             # взвешенная сумма значений

In [5]:
# 🧪 Debug
torch.manual_seed(42)
Q = torch.randn(2, 4, 8)
K = torch.randn(2, 4, 8)
V = torch.randn(2, 4, 8)

out = scaled_dot_product_attention(Q, K, V)
print("Output shape:", out.shape)          # should be (2, 4, 8)
print("Has NaN?    ", torch.isnan(out).any().item())  # should be False
print("Has Inf?    ", torch.isinf(out).any().item())  # should be False

# Cross-attention: seq_q != seq_k
Q2 = torch.randn(1, 3, 16)
K2 = torch.randn(1, 5, 16)
V2 = torch.randn(1, 5, 32)
out2 = scaled_dot_product_attention(Q2, K2, V2)
print("Cross-attn shape:", out2.shape)     # should be (1, 3, 32)

Output shape: torch.Size([2, 4, 8])
Has NaN?     False
Has Inf?     False
Cross-attn shape: torch.Size([1, 3, 32])


In [6]:
# ✅ SUBMIT
from torch_judge import check
check("attention")


🧪 Testing: Softmax Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Output shape (3.1ms)
  ✅ [2/4] Numerical correctness (9.9ms)
  ✅ [3/4] Gradient check (14.8ms)
  ✅ [4/4] Cross-attention (seq_q != seq_k) (0.2ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (28.0ms total)
  Progress saved. Run status() to see your dashboard.



## 6 Multi-Head Attention

In [10]:
import torch
import torch.nn as nn
import math

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

# d_model = размерность эмбеддингов (обычно 256, 512, 1024)
# num_heads = сколько голов внимания (обычно 4, 8, 16)
class MultiHeadAttention:
    def __init__(self, d_model: int, num_heads: int):
        # Initialize W_q, W_k, W_v, W_o
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model,d_model)
        self.W_k = nn.Linear(d_model,d_model)
        self.W_v = nn.Linear(d_model,d_model)
        self.W_o = nn.Linear(d_model,d_model)

    def _split_heads(self,X,B):
        # (B,S,d_model) -> (B,h,S,d_k)
        return X.view(B,-1,self.num_heads,self.d_k).transpose(1,2)
    def forward(self, Q, K, V):
        B = Q.shape[0]
        Qh = self._split_heads(self.W_q(Q),B)
        Kh = self._split_heads(self.W_k(K),B)
        Vh = self._split_heads(self.W_v(V),B)

        scores = Qh @ Kh.transpose(-2,-1) / math.sqrt(self.d_k)
        weights = torch.softmax(scores,dim=-1)
        out = weights @ Vh #(B, h, S_q, d_k)
        return self.W_o(out.transpose(1, 2).contiguous().view(B, -1, self.d_model))

In [24]:
# 🧪 Debug
torch.manual_seed(0)
mha = MultiHeadAttention(d_model=32, num_heads=4)
print("W_q type:", type(mha.W_q))          # should be nn.Linear
print("W_q.weight shape:", mha.W_q.weight.shape)  # (32, 32)

x = torch.randn(2, 6, 32)
out = mha.forward(x, x, x)
print("Output shape:", out.shape)          # (2, 6, 32)

# Cross-attention
Q = torch.randn(1, 3, 32)
K = torch.randn(1, 7, 32)
V = torch.randn(1, 7, 32)
out2 = mha.forward(Q, K, V)
print("Cross-attn shape:", out2.shape)     # (1, 3, 32)

W_q type: <class 'torch.nn.modules.linear.Linear'>
W_q.weight shape: torch.Size([32, 32])
Output shape: torch.Size([2, 6, 32])
Cross-attn shape: torch.Size([1, 3, 32])


In [25]:
# ✅ SUBMIT
from torch_judge import check
check("mha")


🧪 Testing: Multi-Head Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/6] Output shape (4.1ms)
  ✅ [2/6] Uses nn.Linear with correct shapes (0.5ms)
  ✅ [3/6] Numerical correctness vs reference (5.1ms)
  ✅ [4/6] Gradient flow (7.6ms)
  ✅ [5/6] Cross-attention (seq_q != seq_k) (0.9ms)
  ✅ [6/6] Different heads give different outputs (4.2ms)
──────────────────────────────────────────────────
  🎉 All 6 tests passed! (22.3ms total)
  Progress saved. Run status() to see your dashboard.



## 32 Top-k / Top-p (Nucleus) Sampling

In [36]:
def sample_top_k_top_p(logits, top_k=0, top_p=1.0, temperature=1.0):
    if logits.dim() == 1:
        logits = logits.unsqueeze(0)  # (4,) → (1, 4)
    # 1. Temperature
    if temperature != 1:
        logits /= temperature
    
    # 2. Top-K
    if top_k > 0:
        # Находим порог: минимальное значение среди top_k
        threshold = logits.topk(top_k, dim=-1).values[:, -1:]  # (batch, 1)
        
        # Удаляем токены ниже порога
        logits = torch.where(
            logits < threshold,
            torch.full_like(logits, float('-inf')),
            logits
        )
    
    # 3. Top-P 
    if top_p < 1.0:
        # Шаг 1: сортируем логиты по убыванию
        sorted_logits, sorted_indices = logits.sort(descending=True)
        
        # Шаг 2: превращаем в вероятности
        sorted_probs = torch.softmax(sorted_logits, dim=-1)
        
        # Шаг 3: накопленная сумма вероятностей
        cumsum = sorted_probs.cumsum(dim=-1)
        
        # Шаг 4: находим токены, превышающие порог
        sorted_remove = cumsum > top_p
        
        # Шаг 5: сдвигаем вправо (оставляем токен, на котором достигли порога)
        sorted_remove[1:] = sorted_remove[:-1].clone()
        sorted_remove[0] = False  # всегда оставляем хотя бы один токен
        
        # Шаг 6: создаем маску для удаления
        remove = torch.zeros_like(logits, dtype=torch.bool)
        remove.scatter_(1, sorted_indices, sorted_remove)  # ← КЛЮЧЕВАЯ СТРОКА!
        
        # Шаг 7: применяем маску
        logits = logits.masked_fill(remove, float('-inf'))
    print(logits)
    # 4. Выбор токена
    probs = torch.softmax(logits, dim=-1)
    return torch.multinomial(probs, 1).item()

In [37]:
# 🧪 Debug
logits = torch.tensor([[1.0, 5.0, 2.0, 0.5]])  # shape: (1, 4) — 2D!
print('top_k=1:', sample_top_k_top_p(logits.clone(), top_k=1))

print('top_p=0.5:', sample_top_k_top_p(logits.clone(), top_p=0.5))
print('temp=0.01:', sample_top_k_top_p(logits.clone(), temperature=0.01))

tensor([[-inf, 5., -inf, -inf]])
top_k=1: 1
tensor([[1.0000, 5.0000, 2.0000, 0.5000]])
top_p=0.5: 1
tensor([[100., 500., 200.,  50.]])
temp=0.01: 1


In [32]:
# ✅ SUBMIT
from torch_judge import check
check('topk_sampling')


🧪 Testing: Top-k / Top-p Sampling (Medium)
──────────────────────────────────────────────────
  ✅ [1/4] top_k=1 always returns argmax (4.5ms)
  ✅ [2/4] Low temperature concentrates (7.8ms)
  ✅ [3/4] All tokens reachable (no filtering) (446.8ms)
  ✅ [4/4] Returns valid index (5.5ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (464.6ms total)
  Progress saved. Run status() to see your dashboard.

